# ADP 22회 보충 상황형 연습 - 풀이

**상황형 재구성 연습·예시 풀이이며 공식 원문/정답이 아닙니다.**

출처: 기출문제 모음 PDF p.3

원 자료 조건: 원 데이터가 첨부되지 않아 동일 컬럼·분석 주제의 합성 연습 자료를 제공합니다. 수치 정답은 원 시험과 다릅니다.

문제는 동봉 상황형 문제지와 같습니다. 원문은 문항목록.json·추가_문항명세.json에 보존합니다. 순서대로 실행하며 각 대문항에서 고정한 자료·전처리·분할을 후속 문항에 재사용합니다. 통계 답안은 선정 근거 → 가정·가설 → 통계량·p-value → 기각 여부·상황 해석 순서로 작성합니다. 확률·구간 추정·관리도 문항에 불필요한 귀무가설을 만들지 않습니다.

실행: 해당 회차 폴더에서 위에서 아래로 실행합니다. 가설검정_답안.json은 실제 계산한 가설·수치·결론 기록입니다.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC
from IPython.display import display
# repository root 또는 회차 폴더에서 탐색; 경로를 하드코딩하지 않음.
ROOT=next(p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'기출문제/공통/reconstruction.py').exists())
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/'기출문제/공통'))
from reconstruction import *
from matplotlib import font_manager
font_path=ROOT/'기출문제/공통/NanumGothic-Regular.ttf'
font_manager.fontManager.addfont(str(font_path))
plt.rcParams['font.family']='NanumGothic';plt.rcParams['axes.unicode_minus']=False
HERE=ROOT/'기출문제'/'22회/보충';DATA=HERE/'data';OUT=HERE/'실행결과';OUT.mkdir(exist_ok=True)
def load(name):return pd.read_csv(DATA/name)
print('데이터 종류: 합성 복원 연습 자료, 원 시험 데이터가 아님')

# 상황형 답안 기록 초기화
_SCENARIO_ROOT=next(p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'기출문제/공통/scenario_inference.py').is_file())
sys.path.insert(0,str(_SCENARIO_ROOT/'기출문제/공통'))
from scenario_inference import (reset_reports, set_question, report_test, report_anova,
    report_correlations, report_adf, report_residuals, report_regression_diagnostics)
reset_reports(OUT)


데이터 종류: 합성 복원 연습 자료, 원 시험 데이터가 아님


## 1. 제약 아래 생산 계획

재료 a/b/c가 12/12/18개 있다. 제품1은 각 재료 1개씩, 제품2는 각 2개씩 쓴다. 매 단계 만들 수 있는 제품 종류 중 하나를 같은 확률로 골라 재료가 부족할 때까지 생산하는 연습 모형이다.

자료: 데이터 없음

### 3-1

재료를 초과하지 않는 제품1·제품2 수량 조합과 최대 총생산량을 구하시오. 명시한 확률 모형에서 제품1의 기대 생산량과 최대 가능량을 구하고, 최적화와 기대값의 차이를 설명하시오.

**판단 및 풀이 근거**

가능한 수량 범위와 확률을 혼동하지 않습니다. 원문 확률 생성 규칙이 없어 단일 정답은 정의되지 않으며 조건부 예시 기대값만 계산합니다.

**실행 답안**

아래 코드는 이 문항의 산출물과 수치를 계산합니다. 결과와 위 판단 근거를 연결해 답안을 작성합니다.

In [2]:
set_question('3-1')
from functools import lru_cache
feasible=[(x,y) for x in range(13) for y in range(7) if x+2*y<=12]
print('가능 조합:',feasible,'최대 총 제품:',max(x+y for x,y in feasible))
@lru_cache(None)
def expected(a,b,c):
    options=[]
    if min(a,b,c)>=1:options.append(1+expected(a-1,b-1,c-1))
    if min(a,b,c)>=2:options.append(expected(a-2,b-2,c-2))
    return float(np.mean(options)) if options else 0.
print('보완 확률하 제품1 기대개수:',expected(12,12,18),'최대 제품1:',12)

가능 조합: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (5, 0), (5, 1), (5, 2), (5, 3), (6, 0), (6, 1), (6, 2), (6, 3), (7, 0), (7, 1), (7, 2), (8, 0), (8, 1), (8, 2), (9, 0), (9, 1), (10, 0), (10, 1), (11, 0), (12, 0)] 최대 총 제품: 12
보완 확률하 제품1 기대개수: 4.4443359375 최대 제품1: 12


## 2. 상태 발생 순서

한 공정에서 시간순으로 aabbbaaabbbabaaabbab 상태가 관측되었다. 두 상태의 비율이 아니라 상태가 연속해서 나타나는 순서의 무작위성을 조사한다. 유의수준은 5%다.

자료: 데이터 없음

### 4-1

상태의 연속성을 평가할 검정을 선택하고 근거를 설명하시오. 귀무·대립가설, 연속 구간 수·통계량·p-value·기각 여부를 제시하시오. 표본이 짧은 근사 검정의 한계와 단순 비율 비교의 차이를 설명하시오.

**판단 및 풀이 근거**

순서의 무작위성을 평가하는 runs 검정을 선택한다. H0: 주어진 두 상태 개수 하 순서가 무작위, H1: 비무작위다. 짧은 이진 순서의 정규 근사 결과이므로 정확 검정과 차이가 날 수 있다.

**계산·해석 상세**

자료의 시간 순서를 검사합니다. a/b 발생 개수만 가진 chi-square는 이 연속성 귀무가설을 검사하지 않습니다.

**실행 답안**

아래 코드는 이 문항의 산출물과 수치를 계산합니다. 가설·통계량·p-value·기각 여부·상황 결론을 함께 출력하며 수치는 실행 결과에 따라 결정합니다.

In [3]:
set_question('4-1')
from statsmodels.sandbox.stats.runs import runstest_1samp
pattern='aabbbaaabbbabaaabbab';values=np.array([x=='a' for x in pattern],dtype=int)
z,p=runstest_1samp(values,cutoff=.5,correction=True)
print({'runs':1+np.sum(values[1:]!=values[:-1]),'z':z,'p_value':p})

# 상황형 요구: 가설·실제 수치·기각 여부·맥락 결론
report_test('Runs 순서 무작위성 정규 근사','상태 발생 순서가 무작위이다','상태 발생 순서가 비무작위이다',z,p,'공정 상태의 순서가 무작위에서 벗어난다','시간 순서를 보존·이진 상태·짧은 표본 정규 근사 및 연속성 보정의 한계.')

{'runs': np.int64(10), 'z': np.float64(-0.22973414586817037), 'p_value': np.float64(0.8182983598890985)}
{
  "question": "4-1",
  "method": "Runs 순서 무작위성 정규 근사",
  "H0": "상태 발생 순서가 무작위이다",
  "H1": "상태 발생 순서가 비무작위이다",
  "statistic": -0.22973414586817037,
  "p_value": 0.8182983598890985,
  "alpha": 0.05,
  "reject_H0": false,
  "assumptions": "시간 순서를 보존·이진 상태·짧은 표본 정규 근사 및 연속성 보정의 한계.",
  "conclusion": "귀무가설을 기각하지 못한다. 공정 상태의 순서가 무작위에서 벗어난다는 통계적 근거가 충분하지 않다. 귀무가설의 참을 입증한 것은 아니다."
}
